# N2 - The detectability law `R_f*(sigma, Z)` and the Bayes error per `R_f` bin

**Question.** The manuscript claims a critical fault resistance

$$R_f^{\star} \propto \frac{\kappa_{bq}}{\tau\sqrt2\,\sigma_m}$$

(equation `eq_rfstar`, sec. 3.0b), above which snapshot classification degrades towards
chance. Two falsifiable predictions follow: accuracy loss concentrates in the highest `R_f`
bins, and `R_f*` scales as `1/sigma`.

**This notebook tests both.** It also estimates the Bayes error per `R_f` bin with the
Cover-Hart inequality, so the gap between the best classifier and the information-theoretic
floor becomes visible, and it builds the sensor-specification curve (max TVE for a target
accuracy).

**Honest framing.** `eq_rfstar` is a proportionality: it fixes the `1/sigma` exponent and
the linear dependence on `kappa`, but not the constant. So we (a) test the scaling, and
(b) calibrate `kappa` on one feeder and predict the others without refitting.

In [ ]:
# ============================================================================
#  SETUP - run this cell first. Change SMOKE and, if you like, N_JOBS.
# ============================================================================
SMOKE = False         # True: ~1 feeder, 1 seed, small N, minutes. False: full run.

# Optional: restrict this execution to a subset of work units. Leave empty for everything.
# Set e.g. ONLY_FEEDERS = [123] to run just the largest feeder, or ONLY_KS = [2, 3] to run a
# slice of the k grid. Every notebook is resumable, so a sliced run can be continued later
# without redoing finished units (state lives in /kaggle/working/results).
ONLY_FEEDERS = []
ONLY_KS = []

import os, sys, json, time, platform, warnings
warnings.filterwarnings("ignore")
t_start = time.time()

# Where the attached Kaggle dataset is mounted. Everything we need (code, .npz data and
# the precomputed design grids) lives under this one directory.
import glob as _g
_c = sorted(_g.glob("/kaggle/input/**/experiments/k4lib.py", recursive=True))
DATA = os.path.dirname(_c[0]) if _c else "/kaggle/input/snapshot-fd-code-data/experiments"
print("DATA =", DATA)
if not os.path.isdir(DATA):
    # local fallback so the notebook also runs on a laptop for a smoke test
    DATA = os.environ.get("K4_DATA", DATA)
assert os.path.isdir(DATA), (
    "Attach the dataset 'snapshot-fd-code-data' (Add input -> Your datasets). "
    "Expected /kaggle/input/snapshot-fd-code-data/experiments")
# The dataset root also carries k4lib.py and e01.py next to experiments/.
# On Kaggle that is /kaggle/input/snapshot-fd-code-data; locally the task folder.
ROOT = os.environ.get("K4_ROOT", os.path.dirname(DATA))
for p in (DATA, ROOT):
    if p and p not in sys.path:
        sys.path.insert(0, p)

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath(".")
RESULTS = os.path.join(WORK, "results")
os.makedirs(RESULTS, exist_ok=True)
os.environ["K4_RESULTS"] = RESULTS          # k4lib reads this
os.environ["K4_DATA"] = DATA

import numpy as np

# OpenDSS is not needed (design grids are precomputed); placement.py imports simlib,
# which imports opendssdirect at load time. Provide an empty stand-in if it is missing.
try:
    import opendssdirect  # noqa: F401
except ImportError:
    import types as _t
    sys.modules["opendssdirect"] = _t.ModuleType("opendssdirect")

import k4lib as K
import features as F
import placement as P
import placement2 as Q

# forward the optional work-slicing from the SETUP cell into the library
K.ONLY_FEEDERS = list(ONLY_FEEDERS)
K.ONLY_KS = list(ONLY_KS)

print("=== environment ===")
K.print_env()
print("SMOKE =", SMOKE)
if ONLY_FEEDERS:
    print("ONLY_FEEDERS =", ONLY_FEEDERS)
if ONLY_KS:
    print("ONLY_KS =", ONLY_KS)
print("elapsed", round(time.time() - t_start, 1), "s")

## 1. Does `d_b ~ kappa / R_f` actually hold in the data?

This is the assumption the whole law rests on. It is asymptotic in `R_f`: the dip saturates
once the faulted-bus voltage collapses, so `kappa` must be estimated **only** in the region
where `1/R_f` is valid. Estimating it at the smallest `R_f` is wrong by a factor of ~30 —
that error is reproduced and fixed here, so the reader can see why the fit region matters.

In [ ]:
import n2_law as L
from scipy.stats import spearmanr, pearsonr

feeders = K.feeder_list(SMOKE)

print("=== fit quality of d_b ~ kappa/R_f (IEEE-13) ===")
buses, cols, Vp, Vf, meta = K.design_grid(13)
fq = L.fit_quality(cols, Vp, Vf, meta)
print(f"{'R_f':>8} {'measured':>10} {'predicted':>11} {'ratio':>7}")
for row in fq:
    print(f"{row['rf']:8.2f} {row['measured']:10.4f} {row['predicted']:11.4f} {row['ratio']:7.2f}")
print("\nratio ~ 1 above ~1 ohm => the asymptotic law holds there; below it the dip saturates.")

## 2. `R_f*` per fault bus, and does it predict where accuracy collapses?

`kappa_sep` is the class-to-class *difference* of the transfer impedance, because two
classes are separable only while their dip **contrast** exceeds the noise.

In [ ]:
law_rows, bin_rows = [], []

for nb in feeders:
    ds = K.dataset(nb)
    buses, cols, Vp, Vf, meta = K.design_grid(nb)
    kappa, fbs, cls = L.kappa_tables(cols, Vp, Vf, meta)
    ksep = L.separation_kappa(kappa)
    pos = {b: i for i, b in enumerate(fbs)}
    fb_of = np.array([pos.get(b, -1) for b in ds.fbus])

    ks = K.k_list(nb, SMOKE)
    for k in ks:
        for noise in ("1% TVE", "uPMU"):
            ms = K.NOISE[noise][0]
            # sensor set: DMP-2 for this k (the method the manuscript uses)
            _, m2, wgt = Q.margins2(cols, Vp, Vf, meta, mag_sigma=K.DESIGN_NOISE[0], ang_sigma_deg=K.DESIGN_NOISE[1])
            obs, _ = Q.dmp2(m2, wgt, k)
            rfs = L.rf_star(ksep, obs, tau=3.0, mag_sigma=ms)

            X = K.features_of(ds, obs, 0, noise, "FPS")
            seed_kw = dict(seeds=(0,)) if SMOKE else dict(seeds=(0, 1))
            r = K.folds_acc(X, ds.y, ds.rf, return_oof=True, **seed_kw)
            ok = r["oof"] == ds.y

            for b, v in zip(fbs, rfs):
                law_rows.append(dict(feeder=nb, k=k, noise=noise, fault_bus=b,
                                     rf_star=float(v)))
            for lo, hi in [(0, .1), (.1, 1), (1, 5), (5, 20), (20, 100)]:
                m = (ds.rf >= lo) & (ds.rf < hi)
                if not m.sum():
                    continue
                pred = ds.rf[m] <= rfs[fb_of[m]]
                bin_rows.append(dict(feeder=nb, k=k, noise=noise, rf_lo=lo, rf_hi=hi,
                                     n=int(m.sum()),
                                     acc=float(ok[m].mean()), pred=float(pred.mean()),
                                     rf_star_mean=float(rfs[fb_of[m]].mean())))
            print(f"IEEE-{nb} k={k} {noise:8s} acc {r['acc']*100:5.1f}  "
                  f"median R_f* {np.median(rfs):8.2f} ohm")

import pandas as pd
dfb = pd.DataFrame(bin_rows)
print("\n=== accuracy vs predicted detectability by R_f bin ===")
print("(grouped by noise level: pooling the two noise levels would hide the effect,")
print(" since 1% TVE and uPMU have completely different R_f* and accuracy floors)")
for (noise, lo, hi), g in dfb.groupby(["noise", "rf_lo", "rf_hi"]):
    print(f"  {noise:8s} R_f[{lo:5.2f},{hi:5.2f})  n={int(g['n'].sum()):5d}  "
          f"acc {g['acc'].mean()*100:5.1f}%  pred {g['pred'].mean()*100:5.1f}%  "
          f"R_f* {g['rf_star_mean'].mean():8.2f}")
print("\nSpearman(acc, pred) across all bins:",
      round(float(spearmanr(dfb["acc"], dfb["pred"]).statistic), 3))

## 3. The `1/sigma` scaling test

The law says halving the noise doubles `R_f*`. Between the 1 % TVE class and the measured
micro-PMU floor the noise drops by a factor of 50, so `R_f*` should rise by 50x. This is a
genuine prediction, not a fit.

In [ ]:
scale_rows = []
for nb in feeders:
    buses, cols, Vp, Vf, meta = K.design_grid(nb)
    kappa, fbs, cls = L.kappa_tables(cols, Vp, Vf, meta, rf_min_fit=1.0)
    ksep = L.separation_kappa(kappa)
    for k in K.k_list(nb, SMOKE):
        _, m2, wgt = Q.margins2(cols, Vp, Vf, meta, mag_sigma=K.DESIGN_NOISE[0], ang_sigma_deg=K.DESIGN_NOISE[1])
        obs, _ = Q.dmp2(m2, wgt, k)
        med = {}
        for noise, (ms, _) in K.NOISE.items():
            med[noise] = float(np.median(L.rf_star(ksep, obs, tau=3.0, mag_sigma=ms)))
        # R_f* is inversely proportional to sigma, so the uPMU (quieter) set has the
        # LARGER R_f*. The ratio to compare against is therefore uPMU / 1%TVE.
        ratio = med["uPMU"] / med["1% TVE"]
        scale_rows.append(dict(feeder=nb, k=k, rf_star_tve=med["1% TVE"],
                               rf_star_upmu=med["uPMU"], ratio=ratio,
                               sigma_ratio=K.NOISE["1% TVE"][0] / K.NOISE["uPMU"][0]))
        print(f"IEEE-{nb} k={k}: R_f*(1%TVE)={med['1% TVE']:8.2f}  "
              f"R_f*(uPMU)={med['uPMU']:8.2f}  ratio {ratio:6.1f} "
              f"(law predicts {K.NOISE['1% TVE'][0]/K.NOISE['uPMU'][0]:.0f})")
print("\nThe ratio is exact BY CONSTRUCTION of the law (R_f* = kappa/(tau sqrt2 sigma)),")
print("so this cell verifies the implementation, not the physics. The physics test is")
print("cell 2: whether accuracy actually collapses near the predicted R_f*.")

## 4. Bayes error per `R_f` bin (Cover-Hart) and the classifier gap

`features.bayes_lower_bound` estimates the Bayes error from the leave-one-out 1-NN error via
the Cover-Hart inequality (manuscript eq. `eq_ch`). The **gap** (classifier error minus the
Bayes lower bound) says how much of the remaining error is a learning limitation rather than
an information limit.

**A caveat that must be stated, not hidden.** The Cover-Hart bound is a *lower* bound under
its own assumptions, so `gap = clf_err - bayes_lb` should never be negative. In practice it
goes negative at the micro-PMU noise level, which means one of two things: either the 1-NN
estimate is biased upwards in a regime where a 3-class problem is almost perfectly separable
(the bound is vacuous/loose exactly when accuracy -> 100 %), or the mixture assumptions
behind Cover-Hart do not hold for these features. The notebook reports the sign of the gap
rather than trimming it, and the cell flags every violation, because a bound presented as
holding when it does not would be worse than no bound at all.

In [ ]:
bayes_rows = []
for nb in feeders:
    ds = K.dataset(nb)
    buses, cols, Vp, Vf, meta = K.design_grid(nb)
    for k in K.k_list(nb, SMOKE):
        _, m2, wgt = Q.margins2(cols, Vp, Vf, meta, mag_sigma=K.DESIGN_NOISE[0], ang_sigma_deg=K.DESIGN_NOISE[1])
        obs, _ = Q.dmp2(m2, wgt, k)
        for noise in ("1% TVE", "uPMU"):
            X = K.features_of(ds, obs, 0, noise, "FPS")
            for lo, hi in [(0, .1), (.1, 1), (1, 5), (5, 20), (20, 100)]:
                m = (ds.rf >= lo) & (ds.rf < hi)
                if m.sum() < 60:
                    continue
                Xb, yb = X[m], ds.y[m]
                idx = np.random.RandomState(0).choice(len(yb), min(600, len(yb)),
                                                      replace=False) if SMOKE else slice(None)
                e1, lb = F.bayes_lower_bound(Xb[idx], yb[idx], len(ds.classes))
                r = K.folds_acc(Xb, yb, seeds=(0,))
                bayes_rows.append(dict(feeder=nb, k=k, noise=noise, rf_lo=lo, rf_hi=hi,
                                       n=int(m.sum()), nn_err=float(e1),
                                       bayes_lb=float(lb), clf_err=1 - r["acc"],
                                       gap=float((1 - r["acc"]) - lb)))
                print(f"IEEE-{nb} k={k} {noise:8s} R_f[{lo:5.2f},{hi:5.2f}) "
                      f"clf_err {1-r['acc']:.3f}  bayes_lb {lb:.3f}  gap "
                      f"{(1-r['acc'])-lb:+.3f}")

viol = [r for r in bayes_rows if r["gap"] < -1e-9]
print(f"\nCover-Hart violations (gap < 0): {len(viol)} of {len(bayes_rows)} bins")
for noise in ("1% TVE", "uPMU"):
    v = [r for r in viol if r["noise"] == noise]
    print(f"  {noise:8s}: {len(v)} violations")
print("\nInterpretation: the bound is vacuous where accuracy -> 100% (all-classes-"
      "separated regime), which is where the violations concentrate. Report it honestly.")

## 5. Sensor-specification curve

For a target accuracy, what TVE may the sensors have? This is the engineering payoff of the
law: it converts an accuracy target into a procurement specification.

In [ ]:
spec_rows = []
tve_grid = np.array([0.0025, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 3.0])  # percent, P99 TVE class
for nb in feeders:
    ds = K.dataset(nb)
    buses, cols, Vp, Vf, meta = K.design_grid(nb)
    for k in K.k_list(nb, SMOKE):
        _, m2, wgt = Q.margins2(cols, Vp, Vf, meta, mag_sigma=K.DESIGN_NOISE[0], ang_sigma_deg=K.DESIGN_NOISE[1])
        obs, _ = Q.dmp2(m2, wgt, k)
        for tve in tve_grid:
            # FIX (Claude 2026-11-10): the old code swept magnitude only and kept the angle
            # at 0.5 deg (= 0.87 % TVE on its own), so no TVE below ~1 % was ever simulated.
            # Both components now follow the P99 TVE class definition of k4lib.tve_class.
            X = K.features_of(ds, obs, 0, K.tve_class(float(tve)), "FPS")
            r = K.folds_acc(X, ds.y, ds.rf, seeds=(0,))
            spec_rows.append(dict(feeder=nb, k=k, tve_pct=float(tve), acc=r["acc"],
                                  hif=r["hif"]))
            print(f"IEEE-{nb} k={k} TVE {tve:6.3f}%  acc {r['acc']*100:5.1f}%  "
                  f"hif {r['hif']*100:5.1f}%")

import pandas as pd
dfs = pd.DataFrame(spec_rows)
print("\n=== maximum TVE that still meets a target accuracy ===")
for feeder, g in dfs.groupby("feeder"):
    for target in (0.90, 0.95, 0.99):
        okk = g[g["acc"] >= target]
        print(f"IEEE-{feeder}: target {target*100:.0f}% -> "
              f"{'max TVE %.3f%%' % okk['tve_pct'].max() if len(okk) else 'unreachable'}")

## 6. Save

In [ ]:
out = dict(
    law_rows=law_rows, bin_rows=bin_rows, scale_rows=scale_rows,
    bayes_rows=bayes_rows, spec_rows=spec_rows, meta=K.meta())
K.save_unit("n2", {"feeder": "all", "part": "main"}, out)

import pandas as pd
for name, rows in (("N2_rf_star", law_rows), ("N2_bins", bin_rows),
                   ("N2_scaling", scale_rows), ("N2_bayes", bayes_rows),
                   ("N2_spec", spec_rows)):
    pd.DataFrame(rows).to_csv(os.path.join(WORK, name + ".csv"), index=False)
    print("wrote", name + ".csv")
print("TOTAL", round((time.time()-t_start)/60, 1), "min")